# Simplified Advanced RAG Workshop (LlamaIndex)
### Hybrid Search (Dense + BM25) & Cohere Re-ranking in ~15 Lines of Code

This notebook provides a minimal implementation of advanced retrieval techniques using **LlamaIndex**.

## 1. Install Dependencies

In [ ]:
!pip install -q llama-index llama-index-retrievers-bm25 llama-index-postprocessor-cohere-rerank


## 2. Minimal RAG Pipeline

In [ ]:
import os
from llama_index.core import VectorStoreIndex, Document, QueryBundle
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.postprocessor.cohere_rerank import CohereRerank

# Set your Cohere API key for re-ranking (Get a free key at dashboard.cohere.com)
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"

# 1. Sample Dataset Setup
docs = [
    Document(text="Error ERR_502 indicates a gateway timeout when backend service takes over 30s."),
    Document(text="To fix slow PostgreSQL queries, create indexes on foreign keys."),
    Document(text="All API endpoints require JWT authentication expiring after 60 minutes.")
]

# 2. Build Dense & Sparse Retrievers
vector_index = VectorStoreIndex.from_documents(docs)
dense_retriever = vector_index.as_retriever(similarity_top_k=2)
bm25_retriever = BM25Retriever.from_defaults(docstore=vector_index.docstore, similarity_top_k=2)

# 3. Combine into Hybrid Retriever (RRF built-in)
hybrid_retriever = QueryFusionRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    mode="reciprocal_rerank",
    similarity_top_k=3
)

# 4. Add Re-ranker
reranker = CohereRerank(top_n=1)

# --- EXECUTE ---
query = "What causes ERR_502?"

# Step A: Hybrid Search Fetch
candidates = hybrid_retriever.retrieve(query)

# Step B: Re-rank Candidates
final_results = reranker.postprocess_nodes(candidates, query_bundle=QueryBundle(query))

# Print Output
for node in final_results:
    print(f"[Score: {node.score:.4f}] {node.node.get_content()}")
